In [ ]:
# Import the necessary libraries
import ee
import geemap

The Satellite Embedding dataset is a collection of images containing annual images from 2017 onwards.Each image has 64 bands in which each pixel is the incorporation vector representing the time series of various sensors for the given year.


In [2]:
# Trigger the authentication flow.
ee.Authenticate()

# Initialize the library.
ee.Initialize(project='ee-cmcarbajal')

# Create a map
Map = geemap.Map()

In [3]:
# Use the satellite basemap
Map.set_options('SATELLITE')

# Define a geometry to visualize the data
# This geometry covers a new region of interest.
geometry = ee.Geometry.Rectangle([-80.14047121,-6.69765857,-79.17709654,-5.94437422])



# Center the map on the geometry
Map.centerObject(geometry, 10)

In [ ]:
# Prepare the Satellite Embedding dataset
year = 2023
startDate = ee.Date.fromYMD(year, 1, 1)
endDate = startDate.advance(1, 'year')

# Load the dataset
embeddings = ee.ImageCollection('GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL')

# We apply filters and find the images for our chosen year and region
filteredEmbeddings = embeddings \
    .filter(ee.Filter.date(startDate, endDate)) \
    .filter(ee.Filter.bounds(geometry))

In [5]:
# We can use the mosaic() function to combine multiple tiles into a single image.
embeddingImage = filteredEmbeddings.mosaic()

# Print the resulting image to see the bands
print('Satellite Embedding Image', embeddingImage.getInfo())


Satellite Embedding Image {'type': 'Image', 'bands': [{'id': 'A00', 'data_type': {'type': 'PixelType', 'precision': 'double'}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'A01', 'data_type': {'type': 'PixelType', 'precision': 'double'}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'A02', 'data_type': {'type': 'PixelType', 'precision': 'double'}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'A03', 'data_type': {'type': 'PixelType', 'precision': 'double'}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'A04', 'data_type': {'type': 'PixelType', 'precision': 'double'}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'A05', 'data_type': {'type': 'PixelType', 'precision': 'double'}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'A06', 'data_type': {'type': 'PixelType', 'precision': 'double'}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'A07', 'data_type': {'

Visualize the Satellite Embedding dataset. We can visualize the Satellite Embedding dataset using the addLayer() function. We'll use the A01, A16, and A09 bands to create a false color composite.

In [6]:
visParams = {'min': -0.3, 'max': 0.3, 'bands': ['A01', 'A16', 'A09']}
Map.addLayer(embeddingImage.clip(geometry), visParams, 'Embeddings Image')


In [7]:
Map

Map(center=[-6.321146530659889, -79.65878387499988], controls=(WidgetControl(options=['position', 'transparent…

An alternative way to visualize this information is by using it to group pixels with similar embeddings and use these groupings to understand how the model has learnt the spatial and temporal variability of a landscape. We can use unsupervised clustering techniques to group the pixels in 64-dimensional space into groups or "clusters" of similar values. For this, we first sample some pixel values and train an ee.Clusterer.


In [8]:
nSamples = 1000
training = embeddingImage.sample(
    region=geometry,
    scale=10,
    numPixels=nSamples,
    seed=100
)

In [ ]:
# We can now train a clusterer on this sample. Each cluster would represent pixels of similar embeddings.
# Function to train a model for desired number of clusters
def getClusters(nClusters):
    clusterer = ee.Clusterer.wekaKMeans(nClusters=nClusters).train(training)
    # Cluster the image
    clustered = embeddingImage.cluster(clusterer)
    return clustered


# We can also visualize the clusters for 6 clusters
cluster6 = getClusters(6)
Map.addLayer(cluster6.randomVisualizer().clip(geometry), {}, '6 clusters')

# Finally, we can visualize the clusters for 10 clusters
cluster10 = getClusters(10)
Map.addLayer(cluster10.randomVisualizer().clip(geometry), {}, '10 clusters')


In [ ]:
# Export the clustered image to Google Drive
task = ee.batch.Export.image.toDrive(   
    image=cluster10,
    description='Sat_Embedding_Motupe_10Clusters',
    folder='INIA_GEE',
    region=geometry,
    crs='EPSG:32717',
    scale=30,
    fileFormat='GeoTIFF',
    maxPixels=1e13
)
task.start()

In [11]:

# Monitor the task status
task.status()
# Wait for the task to complete
import time
while task.active():
    print('Polling for task (id: {}).'.format(task.id))
    time.sleep(40)  
print('Task completed:', task.status())

Polling for task (id: RSYOYI5RPTXTKNCSIOTEP6CS).
Polling for task (id: RSYOYI5RPTXTKNCSIOTEP6CS).
Polling for task (id: RSYOYI5RPTXTKNCSIOTEP6CS).
Polling for task (id: RSYOYI5RPTXTKNCSIOTEP6CS).
Polling for task (id: RSYOYI5RPTXTKNCSIOTEP6CS).
Polling for task (id: RSYOYI5RPTXTKNCSIOTEP6CS).
Polling for task (id: RSYOYI5RPTXTKNCSIOTEP6CS).
Polling for task (id: RSYOYI5RPTXTKNCSIOTEP6CS).
Polling for task (id: RSYOYI5RPTXTKNCSIOTEP6CS).
Task completed: {'state': 'COMPLETED', 'description': 'Sat_Embedding_Motupe_12Clusters', 'priority': 100, 'creation_timestamp_ms': 1754707264806, 'update_timestamp_ms': 1754707614350, 'start_timestamp_ms': 1754707271517, 'task_type': 'EXPORT_IMAGE', 'destination_uris': ['https://drive.google.com/#folders/1Nq4t2VfINO9Jbx9_bwEVyeuA0eVjz3FJ'], 'attempt': 1, 'batch_eecu_usage_seconds': 550.9986572265625, 'id': 'RSYOYI5RPTXTKNCSIOTEP6CS', 'name': 'projects/ee-cmcarbajal/operations/RSYOYI5RPTXTKNCSIOTEP6CS'}
